### MLflow's Model Registry

#### Interacting with the Model Registry
- In this section We will use the MlflowClient

In [22]:
from mlflow.entities.model_registry import model_version
from mlflow.tracking import MlflowClient
MLFLOW_TRACKING_URI = "sqlite:////home/omar/Documents/sololearn/MLOPS/MLOPSZoomCamp/02-experiment-tracking/mlflow.db"

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)


In [25]:
client.search_experiments()

[<Experiment: artifact_location='/home/omar/Documents/sololearn/MLOPS/MLOPSZoomCamp/02-experiment-tracking/mlruns/2', creation_time=1778174901225, experiment_id='2', last_update_time=1778174901225, lifecycle_stage='active', name='my-cool-experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/home/omar/Documents/sololearn/MLOPS/MLOPSZoomCamp/02-experiment-tracking/mlruns/1', creation_time=1777740957951, experiment_id='1', last_update_time=1777740957951, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1777740405091, experiment_id='0', last_update_time=1777740405091, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

In [24]:
client.create_experiment(name="my-cool-experiment")

'2'

In [26]:
from mlflow.entities import ViewType
runs = client.search_runs(
    experiment_ids=['1'],
    filter_string="",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"],
)

In [28]:
for run in runs:
    print(f"Run id: {run.info.run_id}, run mse: {run.data.metrics['rmse']}")

Run id: b9a05df376584972829c468dfd3e6994, run mse: 5.9698251529653135
Run id: 21839ac88e66465e935f3ed646dd8545, run mse: 5.9698251529653135
Run id: 6760516298b944e8aff6fd869b2d09da, run mse: 5.9698251529653135
Run id: 3f8be55ba95045a1a755f0feabab5d9f, run mse: 5.9698251529653135
Run id: eaf9dbf8acd1408594ec784b3b7295e5, run mse: 5.9698251529653135


In [30]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [31]:
run_id = "b9a05df376584972829c468dfd3e6994"
model_uri = f"runs:/{run_id}/model_xgboost_mflow"

mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor")

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
2026/05/07 12:40:27 WARNING mlflow.tracking._model_registry.fluent: Run with id b9a05df376584972829c468dfd3e6994 has no artifacts at artifact path 'model_xgboost_mflow', registering model based on models:/m-73eb23183a8c467e9a0eb02afe06acec instead
Created version '3' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1778175627987, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1778175627987, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='b9a05df376584972829c468dfd3e6994', run_link=None, source='models:/m-73eb23183a8c467e9a0eb02afe06acec', status='READY', status_message=None, tags={}, user_id=None, version=3, workspace='default'>

In [32]:
client.search_registered_models()

[<RegisteredModel: aliases={'staging': 2}, creation_timestamp=1778173087248, deployment_job_id=None, deployment_job_state=None, description='The NYC Taxi predictor for Trip Duration', last_updated_timestamp=1778175627987, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1778175627987, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1778175627987, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='b9a05df376584972829c468dfd3e6994', run_link=None, source='models:/m-73eb23183a8c467e9a0eb02afe06acec', status='READY', status_message=None, tags={}, user_id=None, version=3, workspace='default'>], name='nyc-taxi-regressor', tags={}, workspace='default'>]

In [36]:
lastest_versions=client.get_latest_versions(name="nyc-taxi-regressor")

/tmp/ipykernel_24782/3487971942.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  lastest_versions=client.get_latest_versions(name="nyc-taxi-regressor")


In [37]:
for version in lastest_versions:
    print(f"version: {version.version}")

version: 3


In [38]:
from datetime import datetime

date = datetime.today().date()
model_version = 3
client.update_model_version(
    name="nyc-taxi-regressor",
    version=model_version,
    description=f"The model version {model_version} was last updated {date}",
)

<ModelVersion: aliases=[], creation_timestamp=1778175627987, current_stage='None', deployment_job_state=None, description='The model version 3 was last updated 2026-05-07', last_updated_timestamp=1778176406344, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='b9a05df376584972829c468dfd3e6994', run_link=None, source='models:/m-73eb23183a8c467e9a0eb02afe06acec', status='READY', status_message=None, tags={}, user_id=None, version=3, workspace='default'>

### Comparing versions and selecting the new "Production" model
In the last section, we will retrieve models registered in the model registry and compare their performance on an unseen test set. The idea is to simulate the scenario in which a deployment engineer has to interact with the model registry to decide whether to update the model version that is in production or not.

These are the steps:

1. Load the test dataset, which corresponds to the NYC Green Taxi data from the month of March 2021.
2. Download the DictVectorizer that was fitted using the training data and saved to MLflow as an artifact, and load it with pickle.
3. Preprocess the test set using the DictVectorizer so we can properly feed the regressors.
4. Make predictions on the test set using the model versions that are currently in the "Staging" and "Production" stages, and compare their performance.
5. Based on the results, update the "Production" model version accordingly.
Note: the model registry doesn't actually deploy the model to production when you transition a model to the "Production" stage, it just assign a label to that model version. You should complement the registry with some CI/CD code that does the actual deployment.

In [56]:
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np

def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(model_uri, X_test, y_test):

    model = mlflow.pyfunc.load_model(model_uri)

    y_pred = model.predict(X_test)

    rmse = np.sqrt(
        mean_squared_error(y_test, y_pred)
    )

    return {"rmse": rmse}

In [42]:
df = read_dataframe("../09-data/green_tripdata_2026-03.parquet")

In [49]:
run_id = "b9a05df376584972829c468dfd3e6994"
model_name = "nyc-taxi-regressor"

In [44]:
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path='.')

'/home/omar/Documents/sololearn/MLOPS/MLOPSZoomCamp/02-experiment-tracking/preprocessor'

In [45]:
import pickle

with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

In [46]:
X_test = preprocess(df, dv)

In [47]:
target = "duration"
y_test = df[target].values

In [57]:
%time test_model(model_uri, X_test=X_test, y_test=y_test)

CPU times: user 2.49 s, sys: 18.1 ms, total: 2.51 s
Wall time: 235 ms


{'rmse': 5.759735498887099}

#### Nota

- stage pronto a deprecarse

In [60]:
client.transition_model_version_stage(
    name=model_name,
    version=3,
    stage="Production",
    archive_existing_versions=True
)

/tmp/ipykernel_24782/684772386.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1778175627987, current_stage='Production', deployment_job_state=None, description='The model version 3 was last updated 2026-05-07', last_updated_timestamp=1778178073574, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='b9a05df376584972829c468dfd3e6994', run_link=None, source='models:/m-73eb23183a8c467e9a0eb02afe06acec', status='READY', status_message=None, tags={}, user_id=None, version=3, workspace='default'>